# 📊 Phase 2.2: The Matrix (Visual Comparison & Matrix Analysis)

## 🏗 Overview: The Scientific Comparison
In this final notebook of Phase 2, we load all our trained models and perform a side-by-side comparison to see which approach best identifies fake news.

### 🔎 Rationale: The Duel of Representations
1.  **TF-IDF**: The 'Accountant'. How well does counting work?
2.  **Word2Vec**: The 'Philosopher'. How well does meaning work?

We use **Dimensionality Reduction (PCA & t-SNE)** to visualize these complex high-dimensional article spaces so we can explain the models to the instructors.

In [ ]:
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

# Colab Integration Check
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print("✅ Running in Colab. Google Drive mounted.")
except ImportError:
    BASE_PATH = '.'
    print("💻 Running Locally.")

## 1. Load All Models & Data
Bringing our pretrained engines back to life.

In [ ]:
test_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/test.csv'))
tfidf_vec = joblib.load(os.path.join(BASE_PATH, 'models/vectorizer.joblib'))
w2v_model = joblib.load(os.path.join(BASE_PATH, 'models/word2vec_model.joblib'))

nb_tfidf = joblib.load(os.path.join(BASE_PATH, 'models/nb_tfidf_classifier.joblib'))
lr_tfidf = joblib.load(os.path.join(BASE_PATH, 'models/lr_tfidf_classifier.joblib'))
lr_w2v = joblib.load(os.path.join(BASE_PATH, 'models/lr_word2vec_classifier.joblib'))

print("✅ All Models Loaded.")

## 2. Dimensionality Reduction Visualization
We take the high-dimensional Word2Vec article vectors and project them into 2D space.

In [ ]:
def document_vector(doc, model):
    words = [w for w in str(doc).split() if w in model.wv.index_to_key]
    if not words: return np.zeros(model.vector_size)
    return np.mean(model.wv[words], axis=0)

# Use a subset to avoid memory crashes in high-dimension visualization
subset_df = test_df.sample(2000, random_state=42)
vectors = np.vstack(subset_df['cleaned_text'].apply(lambda d: document_vector(d, w2v_model)))

# Dimensionality Reduction with PCA
pca = PCA(n_components=2)
viz_pca = pca.fit_transform(vectors)

plt.figure(figsize=(10,6))
sns.scatterplot(x=viz_pca[:,0], y=viz_pca[:,1], hue=subset_df['label'], palette='coolwarm', alpha=0.6)
plt.title('Article Semantic Separation (PCA Reduction)')
plt.legend(title='Category', labels=['Fake', 'Real'])
plt.show()

## 3. Side-by-Side Metadata & Performance Analysis
We compare the metric scores of all three classifiers.

In [ ]:
X_tfidf = tfidf_vec.transform(test_df['cleaned_text'].astype(str))
X_w2v = np.vstack(test_df['cleaned_text'].apply(lambda d: document_vector(d, w2v_model)))
y_test = test_df['label']

models = {
    'NB (TF-IDF)': nb_tfidf.predict(X_tfidf),
    'LR (TF-IDF)': lr_tfidf.predict(X_tfidf),
    'LR (Word2Vec)': lr_w2v.predict(X_w2v)
}

results = []
for name, preds in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'F1-Score': f1_score(y_test, preds)
    })

res_df = pd.DataFrame(results)
print(res_df.sort_values(by='F1-Score', ascending=False))

plt.figure(figsize=(8,5))
sns.barplot(x='Model', y='F1-Score', data=res_df, hue='Model', palette='viridis')
plt.title('Performance Comparison')
plt.ylim(0.8, 1.0)
plt.show()